In [38]:
import pandas as pd

In [57]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Load
df = pd.read_csv("bld/timeline_table.csv")

# Keep only rows where event contains "Final act published"
df1 = (
    df.loc[
        df["event"].str.contains(r"Final act published", case=False, na=False),
        ["procedure", "date", "event", "hyperlink"]
    ]
    .copy()
)

# Extract CELEX number from the hyperlink
df1["celex_number_raw"] = df1["hyperlink"].str.extract(r"numdoc=([A-Z0-9]+)")

# Keep only the main CELEX core
df1["celex_number"] = df1["celex_number_raw"].str.extract(r"^(\d{5}[A-Z]\d{4})")

# Optional: enforce one row per procedure
df1 = (
    df1.sort_values(["procedure", "date"])
       .drop_duplicates(subset="procedure", keep="first")
       .reset_index(drop=True)
)

# Build Eur-Lex URL
df1["eurlex_url"] = df1["celex_number"].apply(
    lambda x: f"https://eur-lex.europa.eu/legal-content/EN/HIS/?uri=CELEX:{x}"
    if pd.notna(x) else None
)

# Function to check whether page is reachable
def check_page_status(url):
    if pd.isna(url):
        return "missing url"
    
    try:
        headers = {
            "User-Agent": "Mozilla/5.0"
        }
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code != 200:
            return f"not reachable ({response.status_code})"
        
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Very simple check: page has a title or body content
        if soup.title or soup.body:
            return "reachable"
        else:
            return "empty page"
    
    except requests.exceptions.RequestException as e:
        return f"error: {type(e).__name__}"

# Add new column
df1["page_status"] = df1["eurlex_url"].apply(check_page_status)

# Print relevant columns
print(df1[["procedure", "celex_number", "eurlex_url", "page_status"]])

df1

          procedure celex_number  \
0    1994/0272(COD)   31996L0057   
1    2000/0073(COD)   32001L0095   
2    2007/0295(COD)   32009R0595   
3    2010/0377(COD)   32012L0018   
4    2010/0380(COD)   32012R0465   
5    2011/0439(COD)   32014L0025   
6    2011/0465(COD)   32014R0332   
7    2012/0358(COD)   32014L0090   
8    2012/0364(COD)   32014R0258   
9    2013/0303(COD)   32014R0546   
10   2014/0011(COD)   32015D1814   
11   2014/0091(COD)   32016L2341   
12   2014/0213(COD)   32015R2102   
13   2015/0284(COD)   32017R1128   
14   2015/0287(COD)   32019L0770   
15  2016/0224B(COD)   32024R1349   
16   2016/0414(COD)   32018L1673   
17   2017/0354(COD)   32019R0515   
18   2018/0298(COD)   32019R0492   
19   2019/0002(COD)   32021R1152   
20   2020/0262(COD)   32022L0431   
21   2020/0347(COD)   32020R2222   
22   2021/0366(COD)   32023R1115   

                                           eurlex_url          page_status  
0   https://eur-lex.europa.eu/legal-content/EN/HIS...  not

,procedure,date,event,hyperlink,celex_number_raw,celex_number,eurlex_url,page_status
0,1994/0272(COD),18/09/1996,Final act published in Official Journal,https://eur-lex.europa.eu/smartapi/cgi/sga_doc...,31996L0057,31996L0057,https://eur-lex.europa.eu/legal-content/EN/HIS...,not reachable (202)
1,2000/0073(COD),15/01/2002,Final act published in Official Journal,https://eur-lex.europa.eu/smartapi/cgi/sga_doc...,32001L0095,32001L0095,https://eur-lex.europa.eu/legal-content/EN/HIS...,not reachable (202)
2,2007/0295(COD),18/07/2009,Final act published in Official Journal,https://eur-lex.europa.eu/smartapi/cgi/sga_doc...,32009R0595R,32009R0595,https://eur-lex.europa.eu/legal-content/EN/HIS...,not reachable (202)
3,2010/0377(COD),24/07/2012,Final act published in Official Journal,https://eur-lex.europa.eu/smartapi/cgi/sga_doc...,32012L0018,32012L0018,https://eur-lex.europa.eu/legal-content/EN/HIS...,not reachable (202)
4,2010/0380(COD),08/06/2012,Final act published in Official Journal,https://eur-lex.europa.eu/smartapi/cgi/sga_doc...,32012R0465,32012R0465,https://eur-lex.europa.eu/legal-content/EN/HIS...,not reachable (202)
5,2011/0439(COD),28/03/2014,Final act published in Official Journal,https://eur-lex.europa.eu/smartapi/cgi/sga_doc...,32014L0025,32014L0025,https://eur-lex.europa.eu/legal-content/EN/HIS...,not reachable (202)
6,2011/0465(COD),05/04/2014,Final act published in Official Journal,https://eur-lex.europa.eu/smartapi/cgi/sga_doc...,32014R0332,32014R0332,https://eur-lex.europa.eu/legal-content/EN/HIS...,not reachable (202)
7,2012/0358(COD),28/08/2014,Final act published in Official Journal,https://eur-lex.europa.eu/smartapi/cgi/sga_doc...,32014L0090R,32014L0090,https://eur-lex.europa.eu/legal-content/EN/HIS...,not reachable (202)
8,2012/0364(COD),08/04/2014,Final act published in Official Journal,https://eur-lex.europa.eu/smartapi/cgi/sga_doc...,32014R0258,32014R0258,https://eur-lex.europa.eu/legal-content/EN/HIS...,not reachable (202)
9,2013/0303(COD),29/05/2014,Final act published in Official Journal,https://eur-lex.europa.eu/smartapi/cgi/sga_doc...,32014R0546,32014R0546,https://eur-lex.europa.eu/legal-content/EN/HIS...,not reachable (202)


In [58]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Load
df = pd.read_csv("bld/timeline_table.csv")

# Keep only rows where event contains "Final act published"
df1 = (
    df.loc[
        df["event"].str.contains(r"Final act published", case=False, na=False),
        ["procedure", "date", "event", "hyperlink"]
    ]
    .copy()
)

# Extract CELEX number from the hyperlink
df1["celex_number_raw"] = df1["hyperlink"].str.extract(r"numdoc=([A-Z0-9]+)")

# Keep only the main CELEX core
df1["celex_number"] = df1["celex_number_raw"].str.extract(r"^(\d{5}[A-Z]\d{4})")

# Optional: enforce one row per procedure
df1 = (
    df1.sort_values(["procedure", "date"])
       .drop_duplicates(subset="procedure", keep="first")
       .reset_index(drop=True)
)

# Build Eur-Lex URL
df1["eurlex_url"] = df1["celex_number"].apply(
    lambda x: f"https://eur-lex.europa.eu/legal-content/EN/HIS/?uri=CELEX:{x}"
    if pd.notna(x) else None
)

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
})

def check_page_status(url):
    if pd.isna(url):
        return "missing_url"

    try:
        r = session.get(url, timeout=15, allow_redirects=True)

        # Detect AWS WAF challenge page
        text = r.text[:5000] if r.text else ""
        if (
            r.status_code == 202
            or "AwsWafIntegration" in text
            or "challenge.js" in text
            or "token.awswaf.com" in text
        ):
            return "waf_challenge"

        if r.status_code != 200:
            return f"http_{r.status_code}"

        soup = BeautifulSoup(r.text, "html.parser")

        # Optional: weak page-content check
        if soup.find("body") is None:
            return "no_body"

        return "reachable"

    except requests.exceptions.RequestException as e:
        return f"error_{type(e).__name__}"

df1["page_status"] = df1["eurlex_url"].apply(check_page_status)

df1[["procedure", "celex_number", "eurlex_url", "page_status"]]

,procedure,celex_number,eurlex_url,page_status
0,1994/0272(COD),31996L0057,https://eur-lex.europa.eu/legal-content/EN/HIS...,waf_challenge
1,2000/0073(COD),32001L0095,https://eur-lex.europa.eu/legal-content/EN/HIS...,waf_challenge
2,2007/0295(COD),32009R0595,https://eur-lex.europa.eu/legal-content/EN/HIS...,waf_challenge
3,2010/0377(COD),32012L0018,https://eur-lex.europa.eu/legal-content/EN/HIS...,waf_challenge
4,2010/0380(COD),32012R0465,https://eur-lex.europa.eu/legal-content/EN/HIS...,waf_challenge
5,2011/0439(COD),32014L0025,https://eur-lex.europa.eu/legal-content/EN/HIS...,waf_challenge
6,2011/0465(COD),32014R0332,https://eur-lex.europa.eu/legal-content/EN/HIS...,waf_challenge
7,2012/0358(COD),32014L0090,https://eur-lex.europa.eu/legal-content/EN/HIS...,waf_challenge
8,2012/0364(COD),32014R0258,https://eur-lex.europa.eu/legal-content/EN/HIS...,waf_challenge
9,2013/0303(COD),32014R0546,https://eur-lex.europa.eu/legal-content/EN/HIS...,waf_challenge


In [59]:
import pandas as pd
from playwright.sync_api import sync_playwright

# assuming df1 already exists and has eurlex_url

def check_with_browser(url):
    if pd.isna(url):
        return "missing_url"
    
    try:
        with sync_playwright() as p:
            browser = p.chromium.launch(headless=True)
            page = browser.new_page()
            page.goto(url, wait_until="domcontentloaded", timeout=30000)

            # wait a bit in case challenge resolves and page reloads
            page.wait_for_timeout(5000)

            final_url = page.url
            html = page.content()

            browser.close()

            if "AwsWafIntegration" in html or "challenge.js" in html:
                return "waf_challenge"

            if "CELEX:" in html or "eur-lex" in final_url.lower():
                return "reachable"

            return "unknown"

    except Exception as e:
        return f"error_{type(e).__name__}"

df1["browser_status"] = df1["eurlex_url"].apply(check_with_browser)
df1[["procedure", "celex_number", "browser_status"]]

,procedure,celex_number,browser_status
0,1994/0272(COD),31996L0057,error_Error
1,2000/0073(COD),32001L0095,error_Error
2,2007/0295(COD),32009R0595,error_Error
3,2010/0377(COD),32012L0018,error_Error
4,2010/0380(COD),32012R0465,error_Error
5,2011/0439(COD),32014L0025,error_Error
6,2011/0465(COD),32014R0332,error_Error
7,2012/0358(COD),32014L0090,error_Error
8,2012/0364(COD),32014R0258,error_Error
9,2013/0303(COD),32014R0546,error_Error


In [61]:
from playwright.sync_api import sync_playwright, Error as PlaywrightError
import traceback

def debug_one_url(url):
    try:
        with sync_playwright() as p:
            browser = p.chromium.launch(
                headless=True,
                chromium_sandbox=False,   # often safer in Docker/Jupyter/root setups
                timeout=60000
            )
            context = browser.new_context(locale="en-GB")
            page = context.new_page()
            page.goto(url, wait_until="domcontentloaded", timeout=60000)
            page.wait_for_timeout(5000)

            print("FINAL URL:", page.url)
            print("TITLE:", page.title())
            print("HTML START:")
            print(page.content()[:1000])

            browser.close()
            return "ok"

    except PlaywrightError as e:
        print("PLAYWRIGHT MESSAGE:")
        print(e.message if hasattr(e, "message") else str(e))
        return f"playwright_error: {e}"

    except Exception as e:
        print("PYTHON ERROR:")
        print(repr(e))
        traceback.print_exc()
        return f"python_error: {e}"

test_url = "https://eur-lex.europa.eu/legal-content/EN/HIS/?uri=CELEX:32023R1115"
debug_one_url(test_url)

PLAYWRIGHT MESSAGE:
It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.


'playwright_error: It looks like you are using Playwright Sync API inside the asyncio loop.\nPlease use the Async API instead.'

In [62]:
from playwright.sync_api import sync_playwright, Error as PlaywrightError
import traceback

def debug_one_url(url):
    try:
        with sync_playwright() as p:
            browser = p.chromium.launch(
                headless=True,
                chromium_sandbox=False,
                timeout=60000
            )
            context = browser.new_context(locale="en-GB")
            page = context.new_page()

            page.on("console", lambda msg: print("CONSOLE:", msg.type, msg.text))
            page.on("requestfailed", lambda req: print("REQUEST FAILED:", req.url, req.failure))

            response = page.goto(url, wait_until="domcontentloaded", timeout=60000)
            page.wait_for_timeout(5000)

            print("STATUS:", response.status if response else None)
            print("FINAL URL:", page.url)
            print("TITLE:", page.title())
            print("HTML START:")
            print(page.content()[:1500])

            browser.close()
            return "ok"

    except PlaywrightError as e:
        print("PLAYWRIGHT MESSAGE:")
        print(e.message if hasattr(e, "message") else str(e))
        return f"playwright_error: {e.message if hasattr(e, 'message') else str(e)}"

    except Exception as e:
        print("PYTHON ERROR:")
        print(repr(e))
        traceback.print_exc()
        return f"python_error: {e}"

test_url = "https://eur-lex.europa.eu/legal-content/EN/HIS/?uri=CELEX:32023R1115"
result = debug_one_url(test_url)
print("RESULT:", result)

PLAYWRIGHT MESSAGE:
It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.
RESULT: playwright_error: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.


In [63]:
from playwright.async_api import async_playwright, Error as PlaywrightError
import traceback

async def debug_one_url(url):
    try:
        async with async_playwright() as p:
            browser = await p.chromium.launch(
                headless=True,
                chromium_sandbox=False,
                timeout=60000
            )
            context = await browser.new_context(locale="en-GB")
            page = await context.new_page()

            page.on("console", lambda msg: print("CONSOLE:", msg.type, msg.text))
            page.on("requestfailed", lambda req: print("REQUEST FAILED:", req.url, req.failure))

            response = await page.goto(url, wait_until="domcontentloaded", timeout=60000)
            await page.wait_for_timeout(5000)

            print("STATUS:", response.status if response else None)
            print("FINAL URL:", page.url)
            print("TITLE:", await page.title())
            print("HTML START:")
            print((await page.content())[:1500])

            await browser.close()
            return "ok"

    except PlaywrightError as e:
        print("PLAYWRIGHT MESSAGE:")
        print(e.message if hasattr(e, "message") else str(e))
        return f"playwright_error: {e.message if hasattr(e, 'message') else str(e)}"

    except Exception as e:
        print("PYTHON ERROR:")
        print(repr(e))
        traceback.print_exc()
        return f"python_error: {e}"

test_url = "https://eur-lex.europa.eu/legal-content/EN/HIS/?uri=CELEX:32023R1115"
result = await debug_one_url(test_url)
print("RESULT:", result)

PYTHON ERROR:
NotImplementedError()
RESULT: python_error: 


Traceback (most recent call last):
  File "C:\Users\User\AppData\Local\Temp\ipykernel_14840\1717281975.py", line 6, in debug_one_url
    async with async_playwright() as p:
  File "c:\Users\User\miniconda3\envs\myenv\Lib\site-packages\playwright\async_api\_context_manager.py", line 46, in __aenter__
    playwright = AsyncPlaywright(next(iter(done)).result())
                                 ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\User\miniconda3\envs\myenv\Lib\site-packages\playwright\_impl\_transport.py", line 120, in connect
    self._proc = await asyncio.create_subprocess_exec(
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\User\miniconda3\envs\myenv\Lib\asyncio\subprocess.py", line 223, in create_subprocess_exec
    transport, protocol = await loop.subprocess_exec(
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\User\miniconda3\envs\myenv\Lib\asyncio\base_events.py", line 1708, in subprocess_exec
    transport = await self._make_su

In [1]:
import pandas as pd

In [22]:
df = pd.read_csv('timeline_table_checked.csv')

In [23]:
import pandas as pd
import requests
import re
import time
from bs4 import BeautifulSoup, Tag
from urllib.parse import urljoin

# -----------------------------
# Patterns
# -----------------------------
COUNCIL_HEADER_PATTERN = re.compile(
    r"""
    (
        Adoption\s+of\s+common\s+position\s+by\s+Council
        |
        Approval\s+of\s+the\s+EP'?s\s+
        (first|second|third)\s+reading\s+position\s+by\s+the\s+Council
        |
        Council\s+agreement
        |
        Discussions\s+within\s+the\s+Council\s+or\s+its\s+preparatory\s+bodies
    )
    """,
    re.I | re.X
)

# matches PRES anywhere in the visible text, including CS-PRES
PRES_PATTERN = re.compile(r'PRES', re.I)

# -----------------------------
# Session
# -----------------------------
session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
})

def find_pres_link_in_container(container, base_url):
    """Return (press_exists, pres_link) from a BS4 container."""
    if container is None:
        return "no", pd.NA

    pres_a = container.find(
        lambda tag: isinstance(tag, Tag)
        and tag.name == "a"
        and PRES_PATTERN.search(tag.get_text(" ", strip=True) or "")
    )

    if pres_a and pres_a.get("href"):
        return "yes", urljoin(base_url, pres_a["href"])

    return "no", pd.NA


def extract_council_adoption_info(url):
    result = {
        "council_adoption_exists": "no",
        "press_exists": "no",
        "pres_link": pd.NA
    }

    try:
        resp = session.get(url, timeout=30)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        # 1) First check literal tables
        target_table = None
        for table in soup.find_all("table"):
            table_text = table.get_text(" ", strip=True)
            if COUNCIL_HEADER_PATTERN.search(table_text):
                target_table = table
                break

        if target_table is not None:
            result["council_adoption_exists"] = "yes"
            result["press_exists"], result["pres_link"] = find_pres_link_in_container(
                target_table, resp.url
            )
            return pd.Series(result)

        # 2) Fallback: look for a section/header/block containing the text
        header_tag = soup.find(
            lambda tag: isinstance(tag, Tag)
            and tag.name in ["h1", "h2", "h3", "h4", "h5", "h6", "summary", "button", "div", "span", "li", "td", "th"]
            and COUNCIL_HEADER_PATTERN.search(tag.get_text(" ", strip=True) or "")
        )

        if header_tag is None:
            return pd.Series(result)

        result["council_adoption_exists"] = "yes"

        candidates = []

        if isinstance(header_tag.parent, Tag):
            candidates.append(header_tag.parent)

        sibling_html = []
        for sib in header_tag.next_siblings:
            if isinstance(sib, Tag) and sib.name in ["h1", "h2", "h3", "h4", "h5", "h6"]:
                break
            sibling_html.append(str(sib))

        if sibling_html:
            sibling_container = BeautifulSoup("".join(sibling_html), "html.parser")
            candidates.append(sibling_container)

        candidates.append(header_tag)

        for candidate in candidates:
            press_exists, pres_link = find_pres_link_in_container(candidate, resp.url)
            if press_exists == "yes":
                result["press_exists"] = "yes"
                result["pres_link"] = pres_link
                break

        return pd.Series(result)

    except requests.RequestException:
        return pd.Series(result)
    except Exception:
        return pd.Series(result)


# -----------------------------
# Apply to dataframe
# -----------------------------
df[["council_adoption_exists", "press_exists", "pres_link"]] = (
    df["eurlex_url"].apply(extract_council_adoption_info)
)

In [24]:
df

,procedure,date,event,hyperlink,celex_number_raw,celex_number,eurlex_url,browser_status,council_adoption_exists,press_exists,pres_link
0,1994/0272(COD),18/09/1996,Final act published in Official Journal,https://eur-lex.europa.eu/smartapi/cgi/sga_doc...,31996L0057,31996L0057,https://eur-lex.europa.eu/legal-content/EN/HIS...,http_202,yes,no,NaN
1,2000/0073(COD),15/01/2002,Final act published in Official Journal,https://eur-lex.europa.eu/smartapi/cgi/sga_doc...,32001L0095,32001L0095,https://eur-lex.europa.eu/legal-content/EN/HIS...,reachable,yes,yes,http://europa.eu/rapid/pressReleasesAction.do?...
2,2007/0295(COD),18/07/2009,Final act published in Official Journal,https://eur-lex.europa.eu/smartapi/cgi/sga_doc...,32009R0595R,32009R0595,https://eur-lex.europa.eu/legal-content/EN/HIS...,reachable,no,no,NaN
3,2010/0377(COD),24/07/2012,Final act published in Official Journal,https://eur-lex.europa.eu/smartapi/cgi/sga_doc...,32012L0018,32012L0018,https://eur-lex.europa.eu/legal-content/EN/HIS...,reachable,yes,yes,http://europa.eu/rapid/pressReleasesAction.do?...
4,2010/0380(COD),08/06/2012,Final act published in Official Journal,https://eur-lex.europa.eu/smartapi/cgi/sga_doc...,32012R0465,32012R0465,https://eur-lex.europa.eu/legal-content/EN/HIS...,reachable,yes,yes,http://europa.eu/rapid/pressReleasesAction.do?...
5,2011/0439(COD),28/03/2014,Final act published in Official Journal,https://eur-lex.europa.eu/smartapi/cgi/sga_doc...,32014L0025,32014L0025,https://eur-lex.europa.eu/legal-content/EN/HIS...,reachable,yes,yes,http://register.consilium.europa.eu/servlet/dr...
6,2011/0465(COD),05/04/2014,Final act published in Official Journal,https://eur-lex.europa.eu/smartapi/cgi/sga_doc...,32014R0332,32014R0332,https://eur-lex.europa.eu/legal-content/EN/HIS...,reachable,yes,yes,http://register.consilium.europa.eu/servlet/dr...
7,2012/0358(COD),28/08/2014,Final act published in Official Journal,https://eur-lex.europa.eu/smartapi/cgi/sga_doc...,32014L0090R,32014L0090,https://eur-lex.europa.eu/legal-content/EN/HIS...,reachable,yes,yes,http://register.consilium.europa.eu/servlet/dr...
8,2012/0364(COD),08/04/2014,Final act published in Official Journal,https://eur-lex.europa.eu/smartapi/cgi/sga_doc...,32014R0258,32014R0258,https://eur-lex.europa.eu/legal-content/EN/HIS...,reachable,yes,yes,http://register.consilium.europa.eu/servlet/dr...
9,2013/0303(COD),29/05/2014,Final act published in Official Journal,https://eur-lex.europa.eu/smartapi/cgi/sga_doc...,32014R0546,32014R0546,https://eur-lex.europa.eu/legal-content/EN/HIS...,reachable,yes,yes,http://register.consilium.europa.eu/servlet/dr...


In [25]:
df.to_csv('pres_con_check.csv', index=False)

In [32]:
import pandas as pd
import requests
import re
from bs4 import BeautifulSoup
from io import BytesIO
from urllib.parse import urlparse, parse_qs
from pypdf import PdfReader

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9"
})

PERSON_START_PATTERN = re.compile(
    r"^(Mr|Mrs|Ms|Mme|M\.|Dr|Prof\.|Professor|Sir|Lord)\b",
    re.I
)

PARTICIPANTS_HEADING_PATTERN = re.compile(r"^PARTICIPANTS(?:\s+\d+)?$", re.I)


def normalize_url(url):
    if pd.isna(url):
        return url
    return str(url).strip().replace("http://", "https://")


def clean_line(x):
    x = x.replace("\xa0", " ")
    x = re.sub(r"[ \t]+", " ", x)
    return x.strip()


def is_separator(line):
    return bool(re.fullmatch(r"(\*+\s*){2,}|\*+\s*\*+\s*\*+", line))


def is_group_header(line):
    return bool(re.fullmatch(r".+?:", line))


def normalize_group(line):
    return re.sub(r"\s*:\s*$", "", line).strip()


def is_all_caps_heading(line):
    if not line:
        return False
    if len(line) > 120:
        return False
    letters = re.sub(r"[^A-Za-z]", "", line)
    if not letters:
        return False
    return letters.upper() == letters


def split_member_designation_same_line(line):
    parts = re.split(r"\t+|\s{2,}", line, maxsplit=1)
    if len(parts) == 2:
        return parts[0].strip(), parts[1].strip()
    return line.strip(), ""


def extract_participants_block_lines(text):
    raw_lines = text.splitlines()
    lines = [clean_line(x) for x in raw_lines]
    lines = [x for x in lines if x]

    participant_idxs = [
        i for i, line in enumerate(lines)
        if PARTICIPANTS_HEADING_PATTERN.match(line)
    ]

    if not participant_idxs:
        return []

    chosen_idx = participant_idxs[-1]
    for idx in participant_idxs:
        window = " ".join(lines[idx+1:idx+12]).lower()
        if "represented as follows" in window:
            chosen_idx = idx
            break

    start = chosen_idx + 1

    block = []
    seen_real_content = False

    for line in lines[start:]:
        if is_all_caps_heading(line) and seen_real_content:
            break

        block.append(line)

        if is_group_header(line) or PERSON_START_PATTERN.match(line):
            seen_real_content = True

    return block


def parse_group_lines(group_lines, group_name, procedure, source_link):
    rows = []

    member_lines = []
    designation_lines = []

    for line in group_lines:
        if not line or is_separator(line):
            continue

        # same-line case: "Mr X    Minister ..."
        if PERSON_START_PATTERN.match(line):
            member, designation = split_member_designation_same_line(line)

            if designation:
                rows.append({
                    "procedure": procedure,
                    "country/other_inst": group_name,
                    "member": member,
                    "designation": designation,
                    "source_link": source_link
                })
            else:
                member_lines.append(member)
        else:
            designation_lines.append(line)

    # pair leftover member-only lines with designation-only lines, in order
    if member_lines:
        if len(designation_lines) >= len(member_lines):
            for i, member in enumerate(member_lines):
                designation = designation_lines[i] if i < len(designation_lines) else ""
                rows.append({
                    "procedure": procedure,
                    "country/other_inst": group_name,
                    "member": member,
                    "designation": designation,
                    "source_link": source_link
                })
        else:
            for member in member_lines:
                rows.append({
                    "procedure": procedure,
                    "country/other_inst": group_name,
                    "member": member,
                    "designation": "",
                    "source_link": source_link
                })

    return rows


def parse_participants_lines(lines, procedure, source_link=None):
    rows = []
    current_group = None
    current_group_lines = []

    def flush_group():
        nonlocal rows, current_group, current_group_lines
        if current_group is not None:
            rows.extend(
                parse_group_lines(
                    current_group_lines,
                    current_group,
                    procedure,
                    source_link
                )
            )
        current_group = None
        current_group_lines = []

    for line in lines:
        if not line:
            continue

        if "represented as follows" in line.lower():
            continue

        if is_all_caps_heading(line) and not PARTICIPANTS_HEADING_PATTERN.match(line):
            flush_group()
            break

        if is_group_header(line):
            flush_group()
            current_group = normalize_group(line)
            continue

        if current_group is not None:
            current_group_lines.append(line)

    flush_group()
    return rows


def get_slug_from_url(url):
    """
    Convert either:
    - https://ec.europa.eu/commission/presscorner/detail/en/pres_01_35
    - https://europa.eu/rapid/pressReleasesAction.do?reference=PRES/01/35&...
    into: pres_01_35
    """
    url = normalize_url(url)

    # case 1: detail URL
    m = re.search(r"/detail/[a-z]{2}/([a-z-]+_\d+_\d+)$", url, re.I)
    if m:
        return m.group(1).lower()

    # case 2: old rapid URL with reference=PRES/01/35 or CS-PRES/12/283
    parsed = urlparse(url)
    q = parse_qs(parsed.query)
    ref = q.get("reference", [None])[0]

    if ref:
        m = re.match(r"([A-Z-]+)/(\d+)/(\d+)", ref, re.I)
        if m:
            prefix = m.group(1).lower()
            yy = m.group(2)
            num = m.group(3)
            return f"{prefix}_{yy}_{num}"

    return None


def build_presscorner_pdf_url(url):
    slug = get_slug_from_url(url)
    if not slug:
        return None

    filename = f"{slug.upper()}_EN.pdf"
    return f"https://ec.europa.eu/commission/presscorner/api/files/document/print/en/{slug}/{filename}"


def get_text_from_html(url):
    url = normalize_url(url)
    resp = session.get(url, timeout=30, allow_redirects=True)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")
    text = soup.get_text("\n", strip=False)

    return text, resp.url


def get_text_from_pdf_url(pdf_url):
    resp = session.get(pdf_url, timeout=30, allow_redirects=True)
    resp.raise_for_status()

    reader = PdfReader(BytesIO(resp.content))
    text = "\n".join(page.extract_text() or "" for page in reader.pages)

    return text, pdf_url


def extract_presscorner_participants(url, procedure, debug=False):
    url = normalize_url(url)

    # 1) try HTML first
    try:
        text, final_url = get_text_from_html(url)
        block_lines = extract_participants_block_lines(text)

        if debug:
            print("\n[HTML]")
            print("procedure:", procedure)
            print("input url :", url)
            print("final url :", final_url)
            print("found     :", bool(block_lines))

        if block_lines:
            return parse_participants_lines(block_lines, procedure, source_link=final_url)

    except Exception as e:
        if debug:
            print("\n[HTML FAILED]")
            print("procedure:", procedure)
            print("url:", url)
            print("error:", repr(e))

    # 2) fallback to print PDF
    try:
        pdf_url = build_presscorner_pdf_url(url)

        if debug:
            print("\n[PDF FALLBACK]")
            print("procedure:", procedure)
            print("pdf url  :", pdf_url)

        if not pdf_url:
            return []

        text, source_link = get_text_from_pdf_url(pdf_url)
        block_lines = extract_participants_block_lines(text)

        if debug:
            print("found     :", bool(block_lines))
            print("first 15 lines:")
            for x in block_lines[:15]:
                print(repr(x))

        if block_lines:
            return parse_participants_lines(block_lines, procedure, source_link=source_link)

        return []

    except Exception as e:
        if debug:
            print("\n[PDF FAILED]")
            print("procedure:", procedure)
            print("url:", url)
            print("error:", repr(e))
        return []

In [33]:
rows = extract_presscorner_participants(
    "https://europa.eu/rapid/pressReleasesAction.do?reference=PRES/01/35&format=HTML&aged=0&lg=en&guiLanguage=en",
    procedure="2000/0073(COD)",
    debug=True
)

pd.DataFrame(rows).head(30)


[HTML]
procedure: 2000/0073(COD)
input url : https://europa.eu/rapid/pressReleasesAction.do?reference=PRES/01/35&format=HTML&aged=0&lg=en&guiLanguage=en
final url : https://ec.europa.eu/commission/presscorner/detail/en/pres_01_35
found     : False

[PDF FALLBACK]
procedure: 2000/0073(COD)
pdf url  : https://ec.europa.eu/commission/presscorner/api/files/document/print/en/pres_01_35/PRES_01_35_EN.pdf
found     : False
first 15 lines:


""


In [35]:
import pandas as pd
import requests
import re
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs, urljoin

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9"
})

def normalize_url(url):
    if pd.isna(url):
        return pd.NA
    return str(url).strip().replace("http://", "https://")

def is_pdf_response(resp):
    ctype = (resp.headers.get("Content-Type") or "").lower()
    final_url = resp.url.lower()
    return (
        "application/pdf" in ctype
        or final_url.endswith(".pdf")
        or final_url.endswith("/pdf")
    )

# ----------------------------
# 1) europa / presscorner -> build EN pdf link
# ----------------------------

def get_presscorner_slug(url):
    """
    Supports:
    - https://ec.europa.eu/commission/presscorner/detail/en/pres_01_35
    - https://europa.eu/rapid/pressReleasesAction.do?reference=PRES/01/35&...
    - https://europa.eu/rapid/pressReleasesAction.do?reference=CS-PRES/12/283&...
    """
    url = normalize_url(url)

    # new/detail form
    m = re.search(r"/detail/[a-z]{2}/([a-z-]+_\d+_\d+)$", url, re.I)
    if m:
        return m.group(1).lower()

    # old rapid form
    parsed = urlparse(url)
    ref = parse_qs(parsed.query).get("reference", [None])[0]

    if ref:
        m = re.match(r"([A-Z-]+)/(\d+)/(\d+)", ref, re.I)
        if m:
            prefix = m.group(1).lower()
            yy = m.group(2)
            num = m.group(3)
            return f"{prefix}_{yy}_{num}"

    return None

def build_presscorner_pdf_url(url):
    slug = get_presscorner_slug(url)
    if not slug:
        return None
    return f"https://ec.europa.eu/commission/presscorner/api/files/document/print/en/{slug}/{slug.upper()}_EN.pdf"

def get_europa_pdf_link(url):
    pdf_url = build_presscorner_pdf_url(url)
    if not pdf_url:
        return False, pd.NA

    try:
        resp = session.get(pdf_url, timeout=30, allow_redirects=True, stream=True)
        resp.raise_for_status()

        if is_pdf_response(resp):
            return True, resp.url

        return False, pd.NA
    except Exception:
        return False, pd.NA

# ----------------------------
# 2) register.consilium -> find EN pdf link
# ----------------------------

def get_consilium_pdf_link(url):
    url = normalize_url(url)

    try:
        resp = session.get(url, timeout=30, allow_redirects=True)
        resp.raise_for_status()

        # sometimes it may already resolve directly to a PDF
        if is_pdf_response(resp):
            return True, resp.url

        soup = BeautifulSoup(resp.text, "html.parser")
        final_url = resp.url

        # best case: explicit EN link to data.consilium.europa.eu ... /en/pdf
        for a in soup.find_all("a", href=True):
            link_text = a.get_text(" ", strip=True)
            href = urljoin(final_url, a["href"])

            if (
                link_text == "EN"
                and "data.consilium.europa.eu" in href.lower()
                and ("/en/pdf" in href.lower() or href.lower().endswith(".pdf"))
            ):
                return True, href

        # fallback: any EN link to data.consilium, then test whether it resolves to pdf
        for a in soup.find_all("a", href=True):
            link_text = a.get_text(" ", strip=True)
            href = urljoin(final_url, a["href"])

            if link_text == "EN" and "data.consilium.europa.eu" in href.lower():
                try:
                    resp2 = session.get(href, timeout=30, allow_redirects=True, stream=True)
                    resp2.raise_for_status()
                    if is_pdf_response(resp2):
                        return True, resp2.url
                except Exception:
                    pass

        return False, pd.NA

    except Exception:
        return False, pd.NA

# ----------------------------
# 3) dispatcher
# ----------------------------

def get_document_pdf_link(pres_link):
    if pd.isna(pres_link):
        return pd.Series({
            "document_found": False,
            "pdf_link": pd.NA
        })

    url = normalize_url(pres_link)
    low = url.lower()

    if "europa.eu" in low or "ec.europa.eu" in low:
        found, pdf_link = get_europa_pdf_link(url)
        return pd.Series({
            "document_found": found,
            "pdf_link": pdf_link
        })

    if "consilium.europa.eu" in low:
        found, pdf_link = get_consilium_pdf_link(url)
        return pd.Series({
            "document_found": found,
            "pdf_link": pdf_link
        })

    return pd.Series({
        "document_found": False,
        "pdf_link": pd.NA
    })

# ----------------------------
# 4) build the new df
# ----------------------------

mask = df["pres_link"].notna()

pdf_df = df.loc[mask, ["procedure", "pres_link"]].copy()

pdf_df[["document_found", "pdf_link"]] = pdf_df["pres_link"].apply(get_document_pdf_link)

pdf_df = pdf_df[["procedure", "document_found", "pdf_link"]]

pdf_df

,procedure,document_found,pdf_link
1,2000/0073(COD),True,https://ec.europa.eu/commission/presscorner/ap...
3,2010/0377(COD),True,https://ec.europa.eu/commission/presscorner/ap...
4,2010/0380(COD),True,https://ec.europa.eu/commission/presscorner/ap...
5,2011/0439(COD),False,NaN
6,2011/0465(COD),False,NaN
7,2012/0358(COD),False,NaN
8,2012/0364(COD),False,NaN
9,2013/0303(COD),False,NaN
10,2014/0011(COD),False,NaN
12,2014/0213(COD),False,NaN


In [36]:
import pandas as pd
import requests
import re
from bs4 import BeautifulSoup
from urllib.parse import urljoin

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9"
})

def normalize_url(url):
    if pd.isna(url):
        return pd.NA
    return str(url).strip().replace("http://", "https://")

def is_pdf_response(resp):
    ctype = (resp.headers.get("Content-Type") or "").lower()
    final_url = resp.url.lower()
    return (
        "application/pdf" in ctype
        or final_url.endswith(".pdf")
        or "/pdf" in final_url
    )

# ----------------------------
# europa logic: keep as-is
# ----------------------------

from urllib.parse import urlparse, parse_qs

def get_presscorner_slug(url):
    url = normalize_url(url)

    m = re.search(r"/detail/[a-z]{2}/([a-z-]+_\d+_\d+)$", url, re.I)
    if m:
        return m.group(1).lower()

    parsed = urlparse(url)
    ref = parse_qs(parsed.query).get("reference", [None])[0]

    if ref:
        m = re.match(r"([A-Z-]+)/(\d+)/(\d+)", ref, re.I)
        if m:
            prefix = m.group(1).lower()
            yy = m.group(2)
            num = m.group(3)
            return f"{prefix}_{yy}_{num}"

    return None

def build_presscorner_pdf_url(url):
    slug = get_presscorner_slug(url)
    if not slug:
        return None
    return f"https://ec.europa.eu/commission/presscorner/api/files/document/print/en/{slug}/{slug.upper()}_EN.pdf"

def get_europa_pdf_link(url):
    pdf_url = build_presscorner_pdf_url(url)
    if not pdf_url:
        return False, pd.NA

    try:
        resp = session.get(pdf_url, timeout=30, allow_redirects=True, stream=True)
        resp.raise_for_status()
        if is_pdf_response(resp):
            return True, resp.url
    except Exception:
        pass

    return False, pd.NA

# ----------------------------
# consilium logic: direct access
# ----------------------------

def get_consilium_pdf_link(url, debug=False):
    url = normalize_url(url)

    try:
        resp = session.get(url, timeout=30, allow_redirects=True)
        resp.raise_for_status()

        if is_pdf_response(resp):
            return True, resp.url

        final_url = resp.url
        soup = BeautifulSoup(resp.text, "html.parser")

        if debug:
            print("\nCONSILIUM")
            print("input url :", url)
            print("final url :", final_url)
            print("status    :", resp.status_code)
            print("title     :", soup.title.get_text(strip=True) if soup.title else None)

        # preferred: exact EN language link to data.consilium
        for a in soup.find_all("a", href=True):
            txt = a.get_text(" ", strip=True)
            href = urljoin(final_url, a["href"])

            if txt == "EN" and "data.consilium.europa.eu" in href.lower():
                return True, href

        # fallback: any href containing data.consilium + /en/
        for a in soup.find_all("a", href=True):
            href = urljoin(final_url, a["href"])
            low = href.lower()

            if "data.consilium.europa.eu" in low and "/en/" in low:
                return True, href

        return False, pd.NA

    except Exception as e:
        if debug:
            print("CONSILIUM FAILED:", url, repr(e))
        return False, pd.NA

# ----------------------------
# dispatcher
# ----------------------------

def get_document_pdf_link(pres_link, debug=False):
    if pd.isna(pres_link):
        return pd.Series({
            "document_found": False,
            "pdf_link": pd.NA
        })

    url = normalize_url(pres_link)
    low = url.lower()

    if "europa.eu" in low or "ec.europa.eu" in low:
        found, pdf_link = get_europa_pdf_link(url)
        return pd.Series({
            "document_found": found,
            "pdf_link": pdf_link
        })

    if "consilium.europa.eu" in low:
        found, pdf_link = get_consilium_pdf_link(url, debug=debug)
        return pd.Series({
            "document_found": found,
            "pdf_link": pdf_link
        })

    return pd.Series({
        "document_found": False,
        "pdf_link": pd.NA
    })

# ----------------------------
# new df
# ----------------------------

pdf_df = df.loc[df["pres_link"].notna(), ["procedure", "pres_link"]].copy()
pdf_df[["document_found", "pdf_link"]] = pdf_df["pres_link"].apply(
    lambda x: get_document_pdf_link(x, debug=False)
)
pdf_df = pdf_df[["procedure", "document_found", "pdf_link"]]

In [37]:
pdf_df

,procedure,document_found,pdf_link
1,2000/0073(COD),True,https://ec.europa.eu/commission/presscorner/ap...
3,2010/0377(COD),True,https://ec.europa.eu/commission/presscorner/ap...
4,2010/0380(COD),True,https://ec.europa.eu/commission/presscorner/ap...
5,2011/0439(COD),False,NaN
6,2011/0465(COD),False,NaN
7,2012/0358(COD),False,NaN
8,2012/0364(COD),False,NaN
9,2013/0303(COD),False,NaN
10,2014/0011(COD),False,NaN
12,2014/0213(COD),False,NaN


In [40]:
import pandas as pd
import requests
import re
from urllib.parse import urlparse, parse_qs

# ---------------------------------------------------
# Session
# ---------------------------------------------------
session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/127.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;q=0.9,"
        "image/avif,image/webp,*/*;q=0.8"
    ),
    "Accept-Language": "en-GB,en;q=0.9",
    "Referer": "https://www.consilium.europa.eu/en/",
    "Upgrade-Insecure-Requests": "1",
    "DNT": "1",
})

# ---------------------------------------------------
# Helpers
# ---------------------------------------------------
def normalize_url(url):
    if pd.isna(url):
        return pd.NA
    return str(url).strip().replace("http://", "https://")

def two_digit_year_to_four_digit(yy):
    """
    13 -> 2013
    96 -> 1996
    Assumes 00-59 => 2000-2059, 60-99 => 1960-1999
    """
    yy = int(yy)
    return 2000 + yy if yy < 60 else 1900 + yy

def extract_docnum_and_year_from_press_link(url):
    """
    From:
    ...ff_COTE_DOCUMENT=10457/13|10457/*/13...
    extract:
      doc_number = '10457'
      yy = '13'
      year = 2013
    """
    url = normalize_url(url)
    if pd.isna(url):
        return pd.NA, pd.NA, pd.NA

    parsed = urlparse(url)
    params = parse_qs(parsed.query)

    cote = params.get("ff_COTE_DOCUMENT", [None])[0]
    if not cote:
        return pd.NA, pd.NA, pd.NA

    # take first chunk before |
    first_part = cote.split("|")[0].strip()

    m = re.match(r"(\d+)/(\d{2})$", first_part)
    if not m:
        return pd.NA, pd.NA, pd.NA

    doc_number = m.group(1)
    yy = m.group(2)
    year = two_digit_year_to_four_digit(yy)

    return doc_number, yy, year

def build_consilium_pdf_candidates(doc_number, year, max_rev=8):
    """
    Build likely direct PDF URLs.
    Example confirmed pattern:
    https://data.consilium.europa.eu/doc/document/ST-10457-2013-REV-1/en/pdf
    """
    if pd.isna(doc_number) or pd.isna(year):
        return []

    base = f"https://data.consilium.europa.eu/doc/document/ST-{doc_number}-{year}"

    candidates = []

    # common forms
    candidates.append(f"{base}-INIT/en/pdf")

    for rev in range(1, max_rev + 1):
        candidates.append(f"{base}-REV-{rev}/en/pdf")

    # optional extra common variants
    for add in range(1, 6):
        candidates.append(f"{base}-ADD-{add}/en/pdf")

    for cor in range(1, 4):
        candidates.append(f"{base}-COR-{cor}/en/pdf")

    # sometimes revisions + addenda/corrigenda exist
    for rev in range(1, max_rev + 1):
        for add in range(1, 4):
            candidates.append(f"{base}-REV-{rev}-ADD-{add}/en/pdf")
        for cor in range(1, 3):
            candidates.append(f"{base}-REV-{rev}-COR-{cor}/en/pdf")

    return candidates

def url_is_pdf(url, timeout=20):
    """
    Check whether URL resolves to a PDF.
    Uses GET(stream=True) rather than HEAD because some sites block HEAD.
    """
    try:
        resp = session.get(url, timeout=timeout, allow_redirects=True, stream=True)
        resp.raise_for_status()

        ctype = (resp.headers.get("Content-Type") or "").lower()
        final_url = resp.url.lower()

        is_pdf = (
            "application/pdf" in ctype
            or final_url.endswith(".pdf")
            or "/pdf" in final_url
        )

        resp.close()

        if is_pdf:
            return True, resp.url
        return False, pd.NA

    except Exception:
        return False, pd.NA

def resolve_register_consilium_pdf(url, max_rev=8, debug=False):
    """
    For a register.consilium press_link:
    1) extract doc number and year from ff_COTE_DOCUMENT
    2) try direct EN PDF URLs
    3) return extracted parts + found link
    """
    doc_number, yy, year = extract_docnum_and_year_from_press_link(url)

    if pd.isna(doc_number):
        return pd.Series({
            "doc_number_extracted": pd.NA,
            "yy_extracted": pd.NA,
            "year_extracted": pd.NA,
            "document_found": False,
            "pdf_link": pd.NA
        })

    candidates = build_consilium_pdf_candidates(doc_number, year, max_rev=max_rev)

    for candidate in candidates:
        found, final_url = url_is_pdf(candidate)

        if debug:
            print(candidate, "->", found)

        if found:
            return pd.Series({
                "doc_number_extracted": doc_number,
                "yy_extracted": yy,
                "year_extracted": year,
                "document_found": True,
                "pdf_link": final_url
            })

    return pd.Series({
        "doc_number_extracted": doc_number,
        "yy_extracted": yy,
        "year_extracted": year,
        "document_found": False,
        "pdf_link": pd.NA
    })

# ---------------------------------------------------
# Apply only to register.consilium rows
# ---------------------------------------------------
register_mask = (
    df["pres_link"].notna()
    & df["pres_link"].astype(str).str.contains(
        "register.consilium.europa.eu",
        case=False,
        regex=False
    )
)

register_pdf_df = df.loc[register_mask, ["procedure", "pres_link"]].copy()

register_pdf_df[
    ["doc_number_extracted", "yy_extracted", "year_extracted", "document_found", "pdf_link"]
] = register_pdf_df["pres_link"].apply(resolve_register_consilium_pdf)

register_pdf_df

,procedure,pres_link,doc_number_extracted,yy_extracted,year_extracted,document_found,pdf_link
5,2011/0439(COD),http://register.consilium.europa.eu/servlet/dr...,6328,14,2014,True,https://data.consilium.europa.eu/doc/document/...
6,2011/0465(COD),http://register.consilium.europa.eu/servlet/dr...,5425,14,2014,True,https://data.consilium.europa.eu/doc/document/...
7,2012/0358(COD),http://register.consilium.europa.eu/servlet/dr...,10457,13,2013,True,https://data.consilium.europa.eu/doc/document/...
8,2012/0364(COD),http://register.consilium.europa.eu/servlet/dr...,8100,14,2014,True,https://data.consilium.europa.eu/doc/document/...
9,2013/0303(COD),http://register.consilium.europa.eu/servlet/dr...,9541,14,2014,True,https://data.consilium.europa.eu/doc/document/...
10,2014/0011(COD),http://register.consilium.europa.eu/servlet/dr...,12166,15,2015,True,https://data.consilium.europa.eu/doc/document/...
12,2014/0213(COD),http://register.consilium.europa.eu/servlet/dr...,8967,15,2015,True,https://data.consilium.europa.eu/doc/document/...


In [43]:
import pandas as pd
import requests
import re
from urllib.parse import urlparse, parse_qs

# ---------------------------------------------------
# Session
# ---------------------------------------------------
session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/127.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;q=0.9,"
        "image/avif,image/webp,*/*;q=0.8"
    ),
    "Accept-Language": "en-GB,en;q=0.9",
    "Referer": "https://www.consilium.europa.eu/en/",
    "Upgrade-Insecure-Requests": "1",
    "DNT": "1",
})

# ---------------------------------------------------
# General helpers
# ---------------------------------------------------
def normalize_url(url):
    if pd.isna(url):
        return pd.NA
    return str(url).strip().replace("http://", "https://")

def is_pdf_response(resp):
    ctype = (resp.headers.get("Content-Type") or "").lower()
    final_url = resp.url.lower()
    return (
        "application/pdf" in ctype
        or final_url.endswith(".pdf")
        or "/pdf" in final_url
    )

# ---------------------------------------------------
# EUROPA logic
# ---------------------------------------------------
def get_presscorner_slug(url):
    url = normalize_url(url)

    # New Press Corner detail URL
    m = re.search(r"/detail/[a-z]{2}/([a-z-]+_\d+_\d+)$", url, re.I)
    if m:
        return m.group(1).lower()

    # Old rapid URL
    parsed = urlparse(url)
    ref = parse_qs(parsed.query).get("reference", [None])[0]

    if ref:
        m = re.match(r"([A-Z-]+)/(\d+)/(\d+)", ref, re.I)
        if m:
            prefix = m.group(1).lower()
            yy = m.group(2)
            num = m.group(3)
            return f"{prefix}_{yy}_{num}"

    return None

def build_presscorner_pdf_url(url):
    slug = get_presscorner_slug(url)
    if not slug:
        return None
    return f"https://ec.europa.eu/commission/presscorner/api/files/document/print/en/{slug}/{slug.upper()}_EN.pdf"

def get_europa_pdf_link(url):
    pdf_url = build_presscorner_pdf_url(url)
    if not pdf_url:
        return False, pd.NA

    try:
        resp = session.get(pdf_url, timeout=30, allow_redirects=True, stream=True)
        resp.raise_for_status()
        if is_pdf_response(resp):
            return True, resp.url
    except Exception:
        pass

    return False, pd.NA

# ---------------------------------------------------
# CONSILIUM logic
# ---------------------------------------------------
def two_digit_year_to_four_digit(yy):
    """
    13 -> 2013
    96 -> 1996
    Assumes:
    00-59 => 2000-2059
    60-99 => 1960-1999
    """
    yy = int(yy)
    return 2000 + yy if yy < 60 else 1900 + yy

def extract_docnum_and_year_from_press_link(url):
    """
    From:
    ...ff_COTE_DOCUMENT=10457/13|10457/*/13...
    extract:
      doc_number = '10457'
      yy = '13'
      year = 2013
    """
    url = normalize_url(url)
    if pd.isna(url):
        return pd.NA, pd.NA, pd.NA

    parsed = urlparse(url)
    params = parse_qs(parsed.query)

    cote = params.get("ff_COTE_DOCUMENT", [None])[0]
    if not cote:
        return pd.NA, pd.NA, pd.NA

    first_part = cote.split("|")[0].strip()

    m = re.match(r"(\d+)/(\d{2})$", first_part)
    if not m:
        return pd.NA, pd.NA, pd.NA

    doc_number = m.group(1)
    yy = m.group(2)
    year = two_digit_year_to_four_digit(yy)

    return doc_number, yy, year

def build_consilium_pdf_candidates(doc_number, year, max_rev=8):
    """
    Example:
    https://data.consilium.europa.eu/doc/document/ST-10457-2013-REV-1/en/pdf
    """
    if pd.isna(doc_number) or pd.isna(year):
        return []

    base = f"https://data.consilium.europa.eu/doc/document/ST-{doc_number}-{year}"
    candidates = []

    # Common variants
    candidates.append(f"{base}-INIT/en/pdf")

    for rev in range(1, max_rev + 1):
        candidates.append(f"{base}-REV-{rev}/en/pdf")

    # Optional extra variants
    for add in range(1, 6):
        candidates.append(f"{base}-ADD-{add}/en/pdf")

    for cor in range(1, 4):
        candidates.append(f"{base}-COR-{cor}/en/pdf")

    for rev in range(1, max_rev + 1):
        for add in range(1, 4):
            candidates.append(f"{base}-REV-{rev}-ADD-{add}/en/pdf")
        for cor in range(1, 3):
            candidates.append(f"{base}-REV-{rev}-COR-{cor}/en/pdf")

    return candidates

def url_is_pdf(url, timeout=20):
    try:
        resp = session.get(url, timeout=timeout, allow_redirects=True, stream=True)
        resp.raise_for_status()

        ctype = (resp.headers.get("Content-Type") or "").lower()
        final_url = resp.url.lower()

        is_pdf = (
            "application/pdf" in ctype
            or final_url.endswith(".pdf")
            or "/pdf" in final_url
        )

        resp.close()

        if is_pdf:
            return True, resp.url
        return False, pd.NA

    except Exception:
        return False, pd.NA

def get_consilium_pdf_link(url, max_rev=8, debug=False):
    """
    For register.consilium links:
    1) extract doc number + year from ff_COTE_DOCUMENT
    2) try likely direct EN PDF URLs
    3) return (document_found, pdf_link)
    """
    doc_number, yy, year = extract_docnum_and_year_from_press_link(url)

    if pd.isna(doc_number):
        return False, pd.NA

    candidates = build_consilium_pdf_candidates(doc_number, year, max_rev=max_rev)

    for candidate in candidates:
        found, final_url = url_is_pdf(candidate)

        if debug:
            print(candidate, "->", found)

        if found:
            return True, final_url

    return False, pd.NA

# ---------------------------------------------------
# Dispatcher
# ---------------------------------------------------
def get_document_pdf_link(pres_link, debug=False):
    if pd.isna(pres_link):
        return pd.Series({
            "document_found": False,
            "pdf_link": pd.NA
        })

    url = normalize_url(pres_link)
    low = url.lower()

    # check consilium FIRST
    if "register.consilium.europa.eu" in low or "consilium.europa.eu" in low:
        found, pdf_link = get_consilium_pdf_link(url, debug=debug)
        return pd.Series({
            "document_found": found,
            "pdf_link": pdf_link
        })

    # then europa / presscorner
    if "ec.europa.eu" in low or "europa.eu/rapid" in low or "commission/presscorner" in low:
        found, pdf_link = get_europa_pdf_link(url)
        return pd.Series({
            "document_found": found,
            "pdf_link": pdf_link
        })

    return pd.Series({
        "document_found": False,
        "pdf_link": pd.NA
    })

# ---------------------------------------------------
# Final output
# ---------------------------------------------------
pdf_df = df.loc[df["pres_link"].notna(), ["procedure", "pres_link"]].copy()

pdf_df[["document_found", "pdf_link"]] = pdf_df["pres_link"].apply(
    lambda x: get_document_pdf_link(x, debug=False)
)

pdf_df = pdf_df[["procedure", "document_found", "pdf_link"]]

pdf_df

,procedure,document_found,pdf_link
1,2000/0073(COD),True,https://ec.europa.eu/commission/presscorner/ap...
3,2010/0377(COD),True,https://ec.europa.eu/commission/presscorner/ap...
4,2010/0380(COD),True,https://ec.europa.eu/commission/presscorner/ap...
5,2011/0439(COD),True,https://data.consilium.europa.eu/doc/document/...
6,2011/0465(COD),True,https://data.consilium.europa.eu/doc/document/...
7,2012/0358(COD),True,https://data.consilium.europa.eu/doc/document/...
8,2012/0364(COD),True,https://data.consilium.europa.eu/doc/document/...
9,2013/0303(COD),True,https://data.consilium.europa.eu/doc/document/...
10,2014/0011(COD),True,https://data.consilium.europa.eu/doc/document/...
12,2014/0213(COD),True,https://data.consilium.europa.eu/doc/document/...


In [44]:
import pandas as pd
import requests
from io import BytesIO
from pypdf import PdfReader

# keep your existing session if already defined
# otherwise:
# session = requests.Session()

def extract_pdf_pages(procedure, link, timeout=40):
    rows = []

    if pd.isna(link):
        return rows

    try:
        resp = session.get(link, timeout=timeout)
        resp.raise_for_status()

        pdf_file = BytesIO(resp.content)
        reader = PdfReader(pdf_file)

        for i, page in enumerate(reader.pages, start=1):
            try:
                text = page.extract_text()
            except Exception:
                text = None

            rows.append({
                "procedure": procedure,
                "link": link,
                "page_num": i,
                "content": text if text is not None else ""
            })

    except Exception as e:
        rows.append({
            "procedure": procedure,
            "link": link,
            "page_num": pd.NA,
            "content": f"[PDF_EXTRACTION_FAILED] {e}"
        })

    return rows


pdf_pages_rows = []

for _, r in pdf_df.loc[pdf_df["document_found"] == True, ["procedure", "pdf_link"]].iterrows():
    pdf_pages_rows.extend(
        extract_pdf_pages(
            procedure=r["procedure"],
            link=r["pdf_link"]
        )
    )

pdf_pages_df = pd.DataFrame(pdf_pages_rows, columns=["procedure", "link", "page_num", "content"])

pdf_pages_df

,procedure,link,page_num,content
0,2000/0073(COD),https://ec.europa.eu/commission/presscorner/ap...,1,/c44/c81/c87/c72/c85/c81/c72/c87/c29/c3/c75/c8...
1,2000/0073(COD),https://ec.europa.eu/commission/presscorner/ap...,2,",,\n5696/01 (Presse 35 - G) 2\n(1\n&21..."
2,2000/0073(COD),https://ec.europa.eu/commission/presscorner/ap...,3,",,\n5696/01 (Presse 35 - G) 3\n(1\n,7(..."
3,2000/0073(COD),https://ec.europa.eu/commission/presscorner/ap...,4,",,\n5696/01 (Presse 35 - G) 4\n(1\n3$5..."
4,2000/0073(COD),https://ec.europa.eu/commission/presscorner/ap...,5,",,\n5696/01 (Presse 35 - G) 5\n(1\n(;&..."
...,...,...,...,...
214,2014/0213(COD),https://data.consilium.europa.eu/doc/document/...,7,19 May 2015 \n \n \n8967/15 7 \n \nEN \n...
215,2014/0213(COD),https://data.consilium.europa.eu/doc/document/...,8,19 May 2015 \n \n \n8967/15 8 \n \nEN \n...
216,2014/0213(COD),https://data.consilium.europa.eu/doc/document/...,9,19 May 2015 \n \n \n8967/15 9 \n \nEN \n...
217,2014/0213(COD),https://data.consilium.europa.eu/doc/document/...,10,19 May 2015 \n \n \n8967/15 10 \n \nEN \...


In [45]:
pdf_pages_df.to_csv("pdf_pages.csv", index=False)

In [50]:
import pandas as pd
import re

# ---------------------------------------------------
# Helpers
# ---------------------------------------------------

PERSON_START_PATTERN = re.compile(
    r"^(Mr|Mrs|Ms|Mme|M\.|Dr|Prof\.|Professor|Sir|Lord)\.?\s*",
    re.I
)

STOP_HEADINGS = {
    "ITEMS DEBATED",
    "OTHER ITEMS APPROVED",
    "ANY OTHER BUSINESS",
    "OTHER BUSINESS",
    "CONTENTS",
    "MAIN RESULTS OF THE COUNCIL",
    "OUTCOME OF THE COUNCIL MEETING",
    "PRESS RELEASE",
    "EVENT IN THE MARGINS OF THE COUNCIL"
}

def clean_line(x):
    if x is None:
        return ""
    x = x.replace("\xa0", " ")
    x = re.sub(r"[ \t]+", " ", x)
    return x.strip()

def is_noise_line(line):
    """
    Remove common PDF header/footer noise.
    """
    if not line:
        return True

    # page/date/header fragments
    if re.fullmatch(r"\d{1,2}\s+[A-Za-z]+\s+\d{4}", line):
        return True
    if re.fullmatch(r"\d{1,2}[./-][IVXLCivxlc.,-]+\d{2,4}", line):
        return True
    if re.fullmatch(r"[A-Z]{2}", line):
        return True
    if re.fullmatch(r"\d+/\d+\s+\d+", line):
        return True
    if re.fullmatch(r"\d+/\d+\s+[IVXLC]+", line):
        return True
    if re.fullmatch(r"[–•\-\s\d]+", line):
        return True

    # pure document/page markers like "9541/14 5"
    if re.fullmatch(r"\d+/\d+\s+\d+", line):
        return True

    return False

def is_separator(line):
    return bool(re.fullmatch(r"(\*+\s*){2,}", line))

def is_group_header(line):
    """
    Country / institution header:
    Belgium:
    Commission:
    High Representative
    The governments of the Acceding States were represented as follows:
    """
    if not line:
        return False

    # standard "Belgium:"
    if line.endswith(":") and not PERSON_START_PATTERN.match(line):
        return True

    return False

def normalize_group(line):
    return re.sub(r"\s*:\s*$", "", line).strip()

def is_stop_heading(line):
    """
    Detect next major section after PARTICIPANTS.
    """
    if not line:
        return False

    line_clean = clean_line(line)
    upper = line_clean.upper()

    if upper in STOP_HEADINGS:
        return True

    # headings in caps
    if (
        len(line_clean) <= 120
        and any(ch.isalpha() for ch in line_clean)
        and line_clean == line_clean.upper()
        and not PERSON_START_PATTERN.match(line_clean)
        and not line_clean.endswith(":")
    ):
        return True

    return False

def split_member_designation_same_line(line):
    """
    Split:
    'Mr Didier REYNDERS Deputy Prime Minister and Minister for Foreign Affairs'
    or tab / many-space separated cases.
    """
    # easiest cases first
    parts = re.split(r"\t+|\s{2,}", line, maxsplit=1)
    if len(parts) == 2:
        return parts[0].strip(), parts[1].strip()

    # fallback:
    # split after a name-like block following title
    m = re.match(
        r"^((?:Mr|Mrs|Ms|Mme|M\.|Dr|Prof\.|Professor|Sir|Lord)\.?\s+.+?)(?=\s+(?:Deputy|Minister|State|Permanent|Vice|Member|Ambassador|President|Prime|Secretary|Commissioner|Chairman|Under|Parliamentary|Federal|Acting|Head|T[aá]naiste|First|Second))",
        line,
        re.I
    )
    if m:
        member = m.group(1).strip()
        designation = line[m.end():].strip()
        return member, designation

    return line.strip(), ""

# ---------------------------------------------------
# Step 1: combine pages per procedure
# ---------------------------------------------------

def combine_pdf_text_by_procedure(pdf_pages_df):
    combined = (
        pdf_pages_df
        .sort_values(["procedure", "page_num"])
        .groupby("procedure", as_index=False)
        .agg(
            link=("link", "first"),
            full_text=("content", lambda x: "\n".join([str(v) for v in x if pd.notna(v)]))
        )
    )
    return combined

# ---------------------------------------------------
# Step 2: extract participants block from full text
# ---------------------------------------------------

def extract_participants_block_lines(full_text):
    raw_lines = full_text.splitlines()
    lines = [clean_line(x) for x in raw_lines]

    # keep lines, but remove very noisy ones
    lines = [x for x in lines if not is_noise_line(x)]

    participant_idxs = [
        i for i, line in enumerate(lines)
        if clean_line(line).upper() == "PARTICIPANTS"
    ]

    intro_phrase_idxs = [
        i for i, line in enumerate(lines)
        if "The governments of the Member States and the European Commission were represented as follows" 
        in line.lower()
    ]

    chosen_idx = None
    start_from_intro_line = False

    # first preference: actual PARTICIPANTS heading
    if participant_idxs:
        for idx in participant_idxs:
            window = lines[idx+1:idx+20]
            joined = " ".join(window).lower()

            if (
                "represented as follows" in joined
                or any(is_group_header(x) for x in window[:10])
                or any(PERSON_START_PATTERN.match(x) for x in window[:10])
            ):
                chosen_idx = idx
                break

        if chosen_idx is None:
            chosen_idx = participant_idxs[-1]

    # fallback: intro sentence without PARTICIPANTS heading
    elif intro_phrase_idxs:
        chosen_idx = intro_phrase_idxs[0]
        start_from_intro_line = True

    else:
        return []

    block = []
    seen_real_content = False

    start_pos = chosen_idx if start_from_intro_line else chosen_idx + 1

    for line in lines[start_pos:]:
        # once content has started, stop at next main heading
        if seen_real_content and is_stop_heading(line):
            break

        block.append(line)

        if is_group_header(line) or PERSON_START_PATTERN.match(line):
            seen_real_content = True

    return [x for x in block if x]

# ---------------------------------------------------
# Step 3: parse participant lines into rows
# ---------------------------------------------------

def parse_participants_lines(lines, procedure):
    rows = []
    current_group = None
    pending_member = None
    pending_designation_parts = []

    def flush_pending():
        nonlocal pending_member, pending_designation_parts, current_group, rows
        if pending_member is not None and current_group is not None:
            designation = " ".join(pending_designation_parts).strip()
            designation = re.sub(r"\s+", " ", designation).strip()

            rows.append({
                "procedure": procedure,
                "country/other_inst": current_group,
                "member": pending_member.strip(),
                "designation": designation
            })
        pending_member = None
        pending_designation_parts = []

    for line in lines:
        if not line:
            continue

        # skip explanatory sentence
        if "represented as follows" in line.lower():
            continue

        # separators like * * *
        if is_separator(line):
            flush_pending()
            continue

        # new country / institution
        if is_group_header(line):
            flush_pending()
            current_group = normalize_group(line)
            continue

        # if a new section accidentally enters block
        if is_stop_heading(line):
            flush_pending()
            break

        # start of a participant
        if PERSON_START_PATTERN.match(line):
            flush_pending()

            member, designation = split_member_designation_same_line(line)
            pending_member = member

            if designation:
                pending_designation_parts = [designation]
            else:
                pending_designation_parts = []

            continue

        # continuation of designation
        if pending_member is not None:
            pending_designation_parts.append(line)
            continue

        # otherwise ignore stray text

    flush_pending()
    return rows

# ---------------------------------------------------
# Step 4: full extraction
# ---------------------------------------------------

combined_pdf_df = combine_pdf_text_by_procedure(pdf_pages_df)

participant_rows = []

for _, r in combined_pdf_df.iterrows():
    block_lines = extract_participants_block_lines(r["full_text"])

    if block_lines:
        participant_rows.extend(
            parse_participants_lines(
                lines=block_lines,
                procedure=r["procedure"]
            )
        )

participants_df = pd.DataFrame(
    participant_rows,
    columns=["procedure", "country/other_inst", "member", "designation"]
)

participants_df

,procedure,country/other_inst,member,designation
0,2010/0377(COD),Belgium,Mr Didier REYNDERS,Deputy Prime Minister and Minister for Foreign...
1,2010/0377(COD),Bulgaria,Mr Nikolaï MLADENOV,Minister for Foreign Affairs
2,2010/0377(COD),Czech Republic,Mr Karel SCHWARZENBERG,First Deputy Prime Minister and Minister for F...
3,2010/0377(COD),Denmark,Mr Nicolai WAMMEN,Minister for European Affairs
4,2010/0377(COD),Germany,Mr Michael LINK,State Minister of the German Ministry of Forei...
...,...,...,...,...
216,2013/0303(COD),Slovakia,Mr Pavol PAVLIS,State Secretary at the Ministry of the Economy
217,2013/0303(COD),Finland,Mr Alexander STUBB,Minister for European Affairs and Foreign Trade
218,2013/0303(COD),Sweden,Ms Ewa BJÖRLING,Minister for Trade and Nordic Cooperation
219,2013/0303(COD),United Kingdom,Mr Ian LIVINGSTON,"Minister of State for Trade and Investment, Fo..."


In [54]:
participants_df.to_csv("participants.csv", index=False)

In [52]:
pdf_df.to_csv("pdf_links.csv", index=False)

In [53]:
import pandas as pd
import re

# ---------------------------------------------------
# Helpers
# ---------------------------------------------------

PERSON_START_PATTERN = re.compile(
    r"^(Mr|Mrs|Ms|Mme|M\.|Dr|Prof\.|Professor|Sir|Lord)\.?\s*",
    re.I
)

STOP_HEADINGS = {
    "ITEMS DEBATED",
    "OTHER ITEMS APPROVED",
    "ANY OTHER BUSINESS",
    "OTHER BUSINESS",
    "CONTENTS",
    "MAIN RESULTS OF THE COUNCIL",
    "OUTCOME OF THE COUNCIL MEETING",
    "PRESS RELEASE",
    "EVENT IN THE MARGINS OF THE COUNCIL"
}

SPECIAL_GROUP_HEADERS = {
    "High Representative",
    "Commission",
    "Other participants",
    "Other Participants"
}

# ---------------------------------------------------
# Text cleaning / recovery
# ---------------------------------------------------

def clean_line(x):
    if x is None:
        return ""
    x = str(x).replace("\xa0", " ")
    x = re.sub(r"[ \t]+", " ", x)
    return x.strip()

def shift_plus_29(text):
    """
    Recover old corrupted PDF text where each char code is shifted down.
    Example:
      3$57,&,3$176 -> PARTICIPANTS
      %HOJLXP     -> Belgium:
    """
    out = []
    for ch in str(text):
        code = ord(ch)
        if 0 <= code <= 255:
            out.append(chr((code + 29) % 256))
        else:
            out.append(ch)
    return "".join(out)

def line_quality_score(line):
    """
    Score how 'readable/useful' a line is.
    Higher score = more likely to be the right recovered version.
    """
    line = clean_line(line)
    if not line:
        return -999

    score = 0

    upper = line.upper()

    # strong signals
    if upper == "PARTICIPANTS":
        score += 50
    if "REPRESENTED AS FOLLOWS" in upper:
        score += 30
    if upper in STOP_HEADINGS:
        score += 20

    # participant / group structure
    if PERSON_START_PATTERN.match(line):
        score += 20
    if line.endswith(":"):
        score += 10

    # readable alphabetic content
    alpha = sum(ch.isalpha() for ch in line)
    weird = sum((not ch.isalnum()) and ch not in " -,:;.'()[]/&" for ch in line)
    score += alpha * 0.2
    score -= weird * 0.7

    # penalise obvious garble patterns
    if re.search(r"[/%$&]{2,}", line):
        score -= 10
    if re.search(r"[A-Z0-9$%&(),./+-]{8,}", line) and alpha < 5:
        score -= 10

    return score

def recover_pdf_line(line):
    """
    Choose between raw line and +29 shifted line.
    """
    raw = clean_line(line)
    dec = clean_line(shift_plus_29(raw))

    raw_score = line_quality_score(raw)
    dec_score = line_quality_score(dec)

    if dec_score > raw_score + 3:
        return dec
    return raw

def recover_pdf_text(full_text):
    """
    Recover text line by line.
    """
    raw_lines = str(full_text).splitlines()
    fixed_lines = [recover_pdf_line(x) for x in raw_lines]
    return "\n".join(fixed_lines)

# ---------------------------------------------------
# Noise / structure detection
# ---------------------------------------------------

def is_noise_line(line):
    line = clean_line(line)
    if not line:
        return True

    # common page/date/header/footer noise
    if re.fullmatch(r"\d{1,2}\s+[A-Za-z]+\s+\d{4}", line):
        return True
    if re.fullmatch(r"\d{1,2}[./-][IVXLCivxlc.,-]+\d{2,4}", line):
        return True
    if re.fullmatch(r"[A-Z]{2}", line):
        return True
    if re.fullmatch(r"\d+/\d+\s+\d+", line):
        return True
    if re.fullmatch(r"\d+/\d+\s+[IVXLC]+", line):
        return True
    if re.fullmatch(r"[–•\-\s\d]+", line):
        return True

    return False

def is_separator(line):
    line = clean_line(line)
    return bool(re.fullmatch(r"(\*+\s*){2,}", line))

def is_stop_heading(line):
    line = clean_line(line)
    if not line:
        return False

    upper = line.upper()

    if upper in STOP_HEADINGS:
        return True

    # all-caps heading, but not participant / country style lines
    if (
        len(line) <= 120
        and any(ch.isalpha() for ch in line)
        and line == line.upper()
        and not PERSON_START_PATTERN.match(line)
        and not line.endswith(":")
    ):
        return True

    return False

def looks_like_group_name(line):
    """
    Detect country / institution names, including cases without a colon.
    """
    line = clean_line(line)
    if not line:
        return False
    if PERSON_START_PATTERN.match(line):
        return False
    if is_stop_heading(line):
        return False

    if line in SPECIAL_GROUP_HEADERS:
        return True

    if line.endswith(":"):
        return True

    # allow country / institution names without colon, e.g. Luxembourg
    if len(line) <= 60 and re.fullmatch(r"[A-Za-zÀ-ÖØ-öø-ÿ'’().,\- ]+", line):
        return True

    return False

def normalize_group(line):
    line = clean_line(line)
    return re.sub(r"\s*:\s*$", "", line).strip()

def split_member_designation_same_line(line):
    """
    Split:
    'Mr Didier REYNDERS Deputy Prime Minister and Minister for Foreign Affairs'
    """
    line = clean_line(line)

    # first: split on multiple spaces / tabs
    parts = re.split(r"\t+|\s{2,}", line, maxsplit=1)
    if len(parts) == 2:
        return parts[0].strip(), parts[1].strip()

    # second: split before common designation starts
    m = re.match(
        r"^((?:Mr|Mrs|Ms|Mme|M\.|Dr|Prof\.|Professor|Sir|Lord)\.?\s+.+?)"
        r"(?=\s+(?:Deputy|Minister|State|Permanent|Vice|Member|Ambassador|President|"
        r"Prime|Secretary|Commissioner|Chairman|Under|Parliamentary|Federal|Acting|"
        r"Head|First|Second|Chancellor|T[aá]naiste))",
        line,
        re.I
    )
    if m:
        member = m.group(1).strip()
        designation = line[m.end():].strip()
        return member, designation

    return line.strip(), ""

def next_nonempty(lines, start_idx):
    for j in range(start_idx + 1, len(lines)):
        if clean_line(lines[j]):
            return clean_line(lines[j])
    return ""

# ---------------------------------------------------
# Step 1: combine pages per procedure
# ---------------------------------------------------

def combine_pdf_text_by_procedure(pdf_pages_df):
    combined = (
        pdf_pages_df
        .sort_values(["procedure", "page_num"])
        .groupby("procedure", as_index=False)
        .agg(
            link=("link", "first"),
            full_text=("content", lambda x: "\n".join([str(v) for v in x if pd.notna(v)]))
        )
    )
    return combined

# ---------------------------------------------------
# Step 2: extract participants block from full text
# ---------------------------------------------------

def extract_participants_block_lines(full_text):
    # first recover bad PDF text if needed
    full_text = recover_pdf_text(full_text)

    raw_lines = full_text.splitlines()
    lines = [clean_line(x) for x in raw_lines]

    # remove very noisy lines
    lines = [x for x in lines if not is_noise_line(x)]

    # primary start: PARTICIPANTS
    participant_idxs = [
        i for i, line in enumerate(lines)
        if clean_line(line).upper() == "PARTICIPANTS"
    ]

    chosen_idx = None

    if participant_idxs:
        for idx in participant_idxs:
            window = lines[idx+1:idx+20]
            joined = " ".join(window).lower()

            if (
                "represented as follows" in joined
                or any(looks_like_group_name(x) for x in window[:10])
                or any(PERSON_START_PATTERN.match(x) for x in window[:10])
            ):
                chosen_idx = idx
                break

        if chosen_idx is None:
            chosen_idx = participant_idxs[-1]

    # fallback: start directly from the intro sentence
    if chosen_idx is None:
        intro_idxs = [
            i for i, line in enumerate(lines)
            if "represented as follows" in line.lower()
        ]
        if intro_idxs:
            chosen_idx = intro_idxs[0] - 1 if intro_idxs[0] > 0 else intro_idxs[0]

    if chosen_idx is None:
        return []

    block = []
    seen_real_content = False

    for line in lines[chosen_idx + 1:]:
        if seen_real_content and is_stop_heading(line):
            break

        block.append(line)

        if looks_like_group_name(line) or PERSON_START_PATTERN.match(line):
            seen_real_content = True

    return [x for x in block if x]

# ---------------------------------------------------
# Step 3: parse participant lines into rows
# ---------------------------------------------------

def parse_participants_lines(lines, procedure):
    rows = []
    current_group = None
    pending_member = None
    pending_designation_parts = []

    def flush_pending():
        nonlocal pending_member, pending_designation_parts, current_group, rows
        if pending_member is not None and current_group is not None:
            designation = " ".join(pending_designation_parts).strip()
            designation = re.sub(r"\s+", " ", designation).strip()

            rows.append({
                "procedure": procedure,
                "country/other_inst": current_group,
                "member": pending_member.strip(),
                "designation": designation
            })
        pending_member = None
        pending_designation_parts = []

    for i, line in enumerate(lines):
        line = clean_line(line)
        if not line:
            continue

        # skip explanatory sentence
        if "represented as follows" in line.lower():
            continue

        # separators like * * *
        if is_separator(line):
            flush_pending()
            continue

        # next section entered accidentally
        if is_stop_heading(line):
            flush_pending()
            break

        # explicit group header
        if looks_like_group_name(line) and (
            line.endswith(":")
            or line in SPECIAL_GROUP_HEADERS
            or (
                pending_member is None
                and PERSON_START_PATTERN.match(next_nonempty(lines, i))
            )
        ):
            flush_pending()
            current_group = normalize_group(line)
            continue

        # participant line
        if PERSON_START_PATTERN.match(line):
            flush_pending()

            member, designation = split_member_designation_same_line(line)
            pending_member = member
            pending_designation_parts = [designation] if designation else []
            continue

        # continuation of designation
        if pending_member is not None:
            pending_designation_parts.append(line)
            continue

        # otherwise ignore stray text

    flush_pending()
    return rows

# ---------------------------------------------------
# Step 4: full extraction
# ---------------------------------------------------

combined_pdf_df = combine_pdf_text_by_procedure(pdf_pages_df)

participant_rows = []

for _, r in combined_pdf_df.iterrows():
    block_lines = extract_participants_block_lines(r["full_text"])

    if block_lines:
        participant_rows.extend(
            parse_participants_lines(
                lines=block_lines,
                procedure=r["procedure"]
            )
        )

participants_df = pd.DataFrame(
    participant_rows,
    columns=["procedure", "country/other_inst", "member", "designation"]
)

participants_df

,procedure,country/other_inst,member,designation
0,2010/0377(COD),Belgium,Mr Didier REYNDERS,Deputy Prime Minister and Minister for Foreign...
1,2010/0377(COD),Bulgaria,Mr Nikolaï MLADENOV,Minister for Foreign Affairs
2,2010/0377(COD),Czech Republic,Mr Karel SCHWARZENBERG,First Deputy Prime Minister and Minister for F...
3,2010/0377(COD),Denmark,Mr Nicolai WAMMEN,Minister for European Affairs
4,2010/0377(COD),Germany,Mr Michael LINK,State Minister of the German Ministry of Forei...
...,...,...,...,...
242,2013/0303(COD),Slovakia,Mr Pavol PAVLIS,State Secretary at the Ministry of the Economy
243,2013/0303(COD),Finland,Mr Alexander STUBB,Minister for European Affairs and Foreign Trade
244,2013/0303(COD),Sweden,Ms Ewa BJÖRLING,Minister for Trade and Nordic Cooperation
245,2013/0303(COD),United Kingdom,Mr Ian LIVINGSTON,"Minister of State for Trade and Investment, Fo..."


In [55]:
import pandas as pd
import re

# ---------------------------------------------------
# Helpers
# ---------------------------------------------------

PERSON_START_PATTERN = re.compile(
    r"^(Mr|Mrs|Ms|Mme|M\.|Dr|Prof\.|Professor|Sir|Lord)\.?\s*",
    re.I
)

STOP_HEADINGS = {
    "ITEMS DEBATED",
    "OTHER ITEMS APPROVED",
    "ANY OTHER BUSINESS",
    "OTHER BUSINESS",
    "CONTENTS",
    "MAIN RESULTS OF THE COUNCIL",
    "OUTCOME OF THE COUNCIL MEETING",
    "PRESS RELEASE",
    "EVENT IN THE MARGINS OF THE COUNCIL"
}

SPECIAL_GROUP_HEADERS = {
    "High Representative",
    "Commission",
    "Other participants",
    "Other Participants",
    "The governments of the Member States and the European Commission were represented as follows",
    "The Governments of the Member States and the European Commission were represented as follows",
    "The governments of the Acceding States were represented as follows",
    "The Governments of the Acceding States were represented as follows",
    "The Government of the Acceding State was represented as follows",
    "The Government of the Acceding State was represented as follows:"
}

def clean_line(x):
    if x is None:
        return ""
    x = str(x).replace("\xa0", " ")
    x = re.sub(r"[ \t]+", " ", x)
    return x.strip()

# ---------------------------------------------------
# Bad old PDF recovery
# ---------------------------------------------------

def shift_plus_29(text):
    """
    Recover old corrupted PDF text where each character code is shifted.
    Example:
      3$57,&,3$176 -> PARTICIPANTS
      %HOJLXP     -> Belgium:
    """
    out = []
    for ch in str(text):
        code = ord(ch)
        if 0 <= code <= 255:
            out.append(chr((code + 29) % 256))
        else:
            out.append(ch)
    return "".join(out)

def line_quality_score(line):
    line = clean_line(line)
    if not line:
        return -999

    score = 0
    upper = line.upper()

    if upper == "PARTICIPANTS":
        score += 50
    if "REPRESENTED AS FOLLOWS" in upper:
        score += 30
    if upper in STOP_HEADINGS:
        score += 15
    if PERSON_START_PATTERN.match(line):
        score += 20
    if line.endswith(":"):
        score += 10

    alpha = sum(ch.isalpha() for ch in line)
    weird = sum((not ch.isalnum()) and ch not in " -,:;.'()[]/&" for ch in line)
    score += alpha * 0.2
    score -= weird * 0.7

    if re.search(r"[/%$&]{2,}", line):
        score -= 10

    return score

def recover_pdf_line(line):
    raw = clean_line(line)
    dec = clean_line(shift_plus_29(raw))

    raw_score = line_quality_score(raw)
    dec_score = line_quality_score(dec)

    if dec_score > raw_score + 3:
        return dec
    return raw

def recover_pdf_text_linewise(full_text):
    raw_lines = str(full_text).splitlines()
    fixed_lines = [recover_pdf_line(x) for x in raw_lines]
    return "\n".join(fixed_lines)

# ---------------------------------------------------
# Structure detection
# ---------------------------------------------------

def is_noise_line(line):
    line = clean_line(line)
    if not line:
        return True

    if re.fullmatch(r"\d{1,2}\s+[A-Za-z]+\s+\d{4}", line):
        return True
    if re.fullmatch(r"\d{1,2}[./-][IVXLCivxlc.,-]+\d{2,4}", line):
        return True
    if re.fullmatch(r"[A-Z]{2}", line):
        return True
    if re.fullmatch(r"\d+/\d+\s+\d+", line):
        return True
    if re.fullmatch(r"\d+/\d+\s+[IVXLC]+", line):
        return True
    if re.fullmatch(r"[–•\-\s\d]+", line):
        return True

    return False

def is_separator(line):
    line = clean_line(line)
    return bool(re.fullmatch(r"(\*+\s*){2,}", line))

def is_stop_heading(line):
    line = clean_line(line)
    if not line:
        return False

    upper = line.upper()

    if upper in STOP_HEADINGS:
        return True

    if (
        len(line) <= 120
        and any(ch.isalpha() for ch in line)
        and line == line.upper()
        and not PERSON_START_PATTERN.match(line)
        and not line.endswith(":")
    ):
        return True

    return False

def looks_like_group_name(line):
    line = clean_line(line)
    if not line:
        return False
    if PERSON_START_PATTERN.match(line):
        return False
    if is_stop_heading(line):
        return False
    if is_separator(line):
        return False

    if line in SPECIAL_GROUP_HEADERS:
        return True

    if line.endswith(":"):
        return True

    # country / institution without colon, e.g. Luxembourg
    if len(line) <= 60 and re.fullmatch(r"[A-Za-zÀ-ÖØ-öø-ÿ'’().,\- ]+", line):
        return True

    return False

def normalize_group(line):
    line = clean_line(line)
    return re.sub(r"\s*:\s*$", "", line).strip()

def split_member_designation_same_line(line):
    line = clean_line(line)

    parts = re.split(r"\t+|\s{2,}", line, maxsplit=1)
    if len(parts) == 2:
        return parts[0].strip(), parts[1].strip()

    m = re.match(
        r"^((?:Mr|Mrs|Ms|Mme|M\.|Dr|Prof\.|Professor|Sir|Lord)\.?\s+.+?)"
        r"(?=\s+(?:Deputy|Minister|State|Permanent|Vice|Member|Ambassador|President|"
        r"Prime|Secretary|Commissioner|Chairman|Under|Parliamentary|Federal|Acting|"
        r"Head|First|Second|Chancellor|T[aá]naiste))",
        line,
        re.I
    )
    if m:
        member = m.group(1).strip()
        designation = line[m.end():].strip()
        return member, designation

    return line.strip(), ""

def next_meaningful_line(lines, start_idx):
    """
    Skip empty lines, noise lines and separators.
    """
    for j in range(start_idx + 1, len(lines)):
        x = clean_line(lines[j])
        if not x:
            continue
        if is_noise_line(x):
            continue
        if is_separator(x):
            continue
        return x
    return ""

# ---------------------------------------------------
# Step 1: combine pages per procedure
# ---------------------------------------------------

def combine_pdf_text_by_procedure(pdf_pages_df):
    return (
        pdf_pages_df
        .sort_values(["procedure", "page_num"])
        .groupby("procedure", as_index=False)
        .agg(
            link=("link", "first"),
            full_text=("content", lambda x: "\n".join([str(v) for v in x if pd.notna(v)]))
        )
    )

# ---------------------------------------------------
# Step 2: extract participants block
# ---------------------------------------------------

def _extract_participants_block_lines_from_text(full_text):
    raw_lines = str(full_text).splitlines()
    lines = [clean_line(x) for x in raw_lines]
    lines = [x for x in lines if not is_noise_line(x)]

    participant_idxs = [
        i for i, line in enumerate(lines)
        if clean_line(line).upper() == "PARTICIPANTS"
    ]

    chosen_idx = None

    if participant_idxs:
        # choose the PARTICIPANTS heading followed by actual participant structure,
        # not the one in the contents page
        for idx in participant_idxs:
            window = lines[idx + 1: idx + 25]
            if (
                any("represented as follows" in x.lower() for x in window)
                or sum(looks_like_group_name(x) for x in window[:15]) >= 2
                or sum(bool(PERSON_START_PATTERN.match(x)) for x in window[:15]) >= 2
            ):
                chosen_idx = idx
                break

        if chosen_idx is None:
            chosen_idx = participant_idxs[-1]

    # fallback: intro sentence directly
    if chosen_idx is None:
        intro_idxs = [
            i for i, line in enumerate(lines)
            if "represented as follows" in line.lower()
        ]
        if intro_idxs:
            chosen_idx = intro_idxs[0] - 1 if intro_idxs[0] > 0 else intro_idxs[0]

    if chosen_idx is None:
        return []

    block = []
    seen_real_content = False

    for line in lines[chosen_idx + 1:]:
        if seen_real_content and is_stop_heading(line):
            break

        block.append(line)

        if looks_like_group_name(line) or PERSON_START_PATTERN.match(line):
            seen_real_content = True

    return [x for x in block if x]

def block_score(block_lines):
    if not block_lines:
        return -999

    score = 0
    score += sum(1 for x in block_lines if looks_like_group_name(x)) * 5
    score += sum(1 for x in block_lines if PERSON_START_PATTERN.match(x)) * 8
    score += sum(1 for x in block_lines if "represented as follows" in x.lower()) * 10
    score -= sum(1 for x in block_lines if is_stop_heading(x)) * 5
    return score

def extract_participants_block_lines(full_text):
    """
    Try multiple recovery strategies and keep the best participant block.
    This is what fixes bad old PDFs like 2000/0073.
    """
    candidates = [
        str(full_text),                          # raw
        recover_pdf_text_linewise(full_text),   # linewise recovery
        shift_plus_29(full_text)                # whole-document recovery
    ]

    best_block = []
    best_score = -999

    for text in candidates:
        block = _extract_participants_block_lines_from_text(text)
        score = block_score(block)
        if score > best_score:
            best_score = score
            best_block = block

    return best_block

# ---------------------------------------------------
# Step 3: parse participant lines into rows
# ---------------------------------------------------

def parse_participants_lines(lines, procedure):
    rows = []
    current_group = None
    pending_member = None
    pending_designation_parts = []

    def flush_pending():
        nonlocal pending_member, pending_designation_parts, current_group, rows
        if pending_member is not None and current_group is not None:
            designation = " ".join(pending_designation_parts).strip()
            designation = re.sub(r"\s+", " ", designation).strip()

            rows.append({
                "procedure": procedure,
                "country/other_inst": current_group,
                "member": pending_member.strip(),
                "designation": designation
            })

        pending_member = None
        pending_designation_parts = []

    for i, line in enumerate(lines):
        line = clean_line(line)
        if not line:
            continue

        if "represented as follows" in line.lower():
            continue

        if is_separator(line):
            flush_pending()
            continue

        if is_stop_heading(line):
            flush_pending()
            break

        # IMPORTANT FIX:
        # detect a new group even if a previous member is still pending
        # this is what fixes Luxembourg being attached to Lithuania
        if looks_like_group_name(line):
            nxt = next_meaningful_line(lines, i)

            if (
                line.endswith(":")
                or line in SPECIAL_GROUP_HEADERS
                or PERSON_START_PATTERN.match(nxt)
            ):
                flush_pending()
                current_group = normalize_group(line)
                continue

        if PERSON_START_PATTERN.match(line):
            flush_pending()
            member, designation = split_member_designation_same_line(line)
            pending_member = member
            pending_designation_parts = [designation] if designation else []
            continue

        if pending_member is not None:
            pending_designation_parts.append(line)
            continue

    flush_pending()
    return rows

# ---------------------------------------------------
# Step 4: full extraction
# ---------------------------------------------------

combined_pdf_df = combine_pdf_text_by_procedure(pdf_pages_df)

participant_rows = []

for _, r in combined_pdf_df.iterrows():
    block_lines = extract_participants_block_lines(r["full_text"])

    if block_lines:
        participant_rows.extend(
            parse_participants_lines(
                lines=block_lines,
                procedure=r["procedure"]
            )
        )

participants_df = pd.DataFrame(
    participant_rows,
    columns=["procedure", "country/other_inst", "member", "designation"]
)

participants_df

,procedure,country/other_inst,member,designation
0,2010/0377(COD),Belgium,Mr Didier REYNDERS,Deputy Prime Minister and Minister for Foreign...
1,2010/0377(COD),Bulgaria,Mr Nikolaï MLADENOV,Minister for Foreign Affairs
2,2010/0377(COD),Czech Republic,Mr Karel SCHWARZENBERG,First Deputy Prime Minister and Minister for F...
3,2010/0377(COD),Denmark,Mr Nicolai WAMMEN,Minister for European Affairs
4,2010/0377(COD),Germany,Mr Michael LINK,State Minister of the German Ministry of Forei...
...,...,...,...,...
217,2013/0303(COD),Slovakia,Mr Pavol PAVLIS,State Secretary at the Ministry of the Economy
218,2013/0303(COD),Finland,Mr Alexander STUBB,Minister for European Affairs and Foreign Trade
219,2013/0303(COD),Sweden,Ms Ewa BJÖRLING,Minister for Trade and Nordic Cooperation
220,2013/0303(COD),United Kingdom,Mr Ian LIVINGSTON,"Minister of State for Trade and Investment, Fo..."
